# 第15章：在线服务与实盘

## 本章学习目标

- 理解在线服务架构
- 掌握实时数据更新
- 学会策略部署
- 了解风险管理

---

## 15.1 在线服务概述

将训练好的模型部署到生产环境，需要考虑多个方面的问题。

### 架构图

```
┌─────────────────────────────────────────────────────────────┐
│                    在线服务架构                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐  │
│  │ 数据源       │ →  │ 数据处理     │ →  │ 特征计算     │  │
│  │ (实时行情)   │    │ (清洗对齐)   │    │ (Alpha因子)  │  │
│  └──────────────┘    └──────────────┘    └──────────────┘  │
│         ↓                   ↓                   ↓          │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐  │
│  │ 模型推理     │ →  │ 信号生成     │ →  │ 订单执行     │  │
│  │ (预测评分)   │    │ (选股策略)   │    │ (交易接口)   │  │
│  └──────────────┘    └──────────────┘    └──────────────┘  │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 关键组件

| 组件 | 功能 | 注意事项 |
|------|------|----------|
| 数据源 | 获取实时数据 | 延迟、可靠性 |
| 特征计算 | 计算因子 | 性能、一致性 |
| 模型推理 | 预测评分 | 延迟、内存 |
| 信号生成 | 选股决策 | 逻辑验证 |
| 订单执行 | 发送交易 | 风控、成本 |

In [ ]:
import qlib
from qlib.workflow import R
from qlib.workflow.online import OnlineManager
import pandas as pd
import numpy as np

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 15.2 模型管理

In [ ]:
# 模型版本管理
print("模型管理最佳实践:")
print("=" * 50)

practices = {
    "版本控制": "使用语义化版本号 (v1.0.0)",
    "模型存储": "保存模型参数、配置、训练数据",
    "A/B测试": "同时运行多个模型版本",
    "回滚机制": "支持快速回滚到上一版本",
    "监控告警": "监控模型性能指标",
}

for practice, desc in practices.items():
    print(f"\n{practice}:")
    print(f"  {desc}")

In [ ]:
# 使用 Recorder 管理模型
from qlib.workflow import R

# 加载已保存的模型
def load_model_from_recorder(experiment_name, recorder_id=None):
    """
    从 Recorder 加载模型
    
    参数:
        experiment_name: 实验名称
        recorder_id: Recorder ID (可选，默认使用最新的)
    """
    # 获取所有 recorder
    recorders = R.list_recorders(experiment_name=experiment_name)
    
    if not recorders:
        print(f"实验 {experiment_name} 没有找到 Recorder")
        return None
    
    # 选择 recorder
    if recorder_id is None:
        recorder_id = list(recorders.keys())[0]
    
    recorder = recorders[recorder_id]
    
    # 加载模型
    model = recorder.load_object("params.pkl")  # 示例
    
    print(f"从 Recorder {recorder_id} 加载模型")
    return model

print("模型加载函数定义完成")

## 15.3 数据更新

In [ ]:
# 数据更新策略
print("数据更新策略:")
print("=" * 50)

strategies = {
    "定时更新": "每天收盘后更新数据",
    "增量更新": "只更新新增数据，节省时间",
    "校验机制": "检查数据完整性和正确性",
    "异常处理": "处理数据缺失和异常值",
}

for strategy, desc in strategies.items():
    print(f"\n{strategy}:")
    print(f"  {desc}")

In [ ]:
# 模拟数据更新流程
class DataUpdater:
    """数据更新器"""
    
    def __init__(self, provider_uri):
        self.provider_uri = provider_uri
        self.last_update = None
    
    def check_update_needed(self):
        """检查是否需要更新"""
        # 获取最新交易日
        from qlib.data import D
        calendar = D.calendar(freq="day")
        latest_trade_date = calendar[-1]
        
        return latest_trade_date
    
    def update_data(self):
        """更新数据"""
        print("开始更新数据...")
        
        # 1. 检查更新
        latest_date = self.check_update_needed()
        print(f"最新交易日: {latest_date}")
        
        # 2. 下载数据
        # 这里需要根据实际数据源配置
        print("数据下载中...")
        
        # 3. 数据校验
        print("数据校验中...")
        
        # 4. 更新完成
        self.last_update = latest_date
        print(f"数据更新完成: {latest_date}")

# 创建数据更新器
updater = DataUpdater("~/.qlib/qlib_data/cn_data")
updater.update_data()

## 15.4 在线推理服务

In [ ]:
# 在线推理服务示例
class OnlineInferenceService:
    """在线推理服务"""
    
    def __init__(self, model, handler_config):
        """
        参数:
            model: 训练好的模型
            handler_config: 数据处理配置
        """
        self.model = model
        self.handler_config = handler_config
    
    def get_latest_features(self, instruments):
        """获取最新特征"""
        from qlib.data import D
        
        # 获取最近的交易日
        calendar = D.calendar(freq="day")
        end_date = calendar[-1]
        start_date = calendar[-60]  # 取60天数据计算特征
        
        # 获取特征数据
        features = D.features(
            instruments=instruments,
            fields=self.handler_config['fields'],
            start_time=start_date,
            end_time=end_date,
        )
        
        return features
    
    def predict(self, instruments):
        """预测"""
        features = self.get_latest_features(instruments)
        
        # 数据预处理
        # ...
        
        # 模型预测
        predictions = self.model.predict(features)
        
        return predictions

print("在线推理服务定义完成")

## 15.5 风险控制

In [ ]:
# 风险控制措施
print("风险控制措施:")
print("=" * 50)

risk_controls = {
    "仓位限制": "单只股票仓位上限",
    "行业限制": "单一行业仓位上限",
    "止损机制": "达到止损线自动平仓",
    "异常检测": "检测异常交易和信号",
    "熔断机制": "市场异常时暂停交易",
    "模型监控": "监控模型预测质量",
}

for control, desc in risk_controls.items():
    print(f"\n{control}:")
    print(f"  {desc}")

In [ ]:
# 简单的风险检查器
class RiskChecker:
    """风险检查器"""
    
    def __init__(self, 
                 max_position_pct=0.05,
                 max_sector_pct=0.30,
                 max_drawdown=0.10):
        self.max_position_pct = max_position_pct
        self.max_sector_pct = max_sector_pct
        self.max_drawdown = max_drawdown
    
    def check_position_limit(self, positions):
        """检查仓位限制"""
        violations = []
        
        for stock, weight in positions.items():
            if weight > self.max_position_pct:
                violations.append({
                    'type': 'position_limit',
                    'stock': stock,
                    'weight': weight,
                    'limit': self.max_position_pct,
                })
        
        return violations
    
    def check_drawdown(self, current_value, peak_value):
        """检查回撤"""
        drawdown = (peak_value - current_value) / peak_value
        
        if drawdown > self.max_drawdown:
            return {
                'type': 'drawdown_limit',
                'drawdown': drawdown,
                'limit': self.max_drawdown,
            }
        return None

# 创建风险检查器
risk_checker = RiskChecker()

# 测试
test_positions = {'SH600000': 0.08, 'SH600016': 0.03}
violations = risk_checker.check_position_limit(test_positions)

if violations:
    print("\n风险违规:")
    for v in violations:
        print(f"  {v}")
else:
    print("\n风险检查通过")

## 15.6 监控与告警

In [ ]:
# 监控指标
print("监控指标:")
print("=" * 50)

monitoring_metrics = {
    "系统指标": ["CPU使用率", "内存使用率", "响应延迟"],
    "模型指标": ["预测IC", "预测分布", "特征重要性变化"],
    "交易指标": ["成交率", "滑点", "换手率"],
    "收益指标": ["日收益", "累计收益", "最大回撤"],
}

for category, metrics in monitoring_metrics.items():
    print(f"\n{category}:")
    for m in metrics:
        print(f"  - {m}")

## 15.7 部署清单

In [ ]:
# 部署检查清单
deployment_checklist = {
    "模型": [
        "模型已训练并保存",
        "模型在测试集表现达标",
        "模型版本已记录",
    ],
    "数据": [
        "数据源配置正确",
        "数据更新流程测试通过",
        "数据校验机制就绪",
    ],
    "系统": [
        "服务部署完成",
        "监控告警配置",
        "日志系统就绪",
    ],
    "风控": [
        "风险限制配置",
        "异常处理机制",
        "应急方案准备",
    ],
}

print("部署检查清单:")
print("=" * 50)

for category, items in deployment_checklist.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  [ ] {item}")

## 15.8 实践建议

In [ ]:
# 实践建议
print("实盘部署建议:")
print("=" * 60)

recommendations = [
    "1. 从小资金开始，逐步放大规模",
    "2. 先在模拟环境充分测试",
    "3. 建立完善的监控和告警系统",
    "4. 准备应急处理方案",
    "5. 定期回顾和优化策略",
    "6. 保持模型定期重训",
    "7. 记录所有交易和决策",
    "8. 关注市场变化和模型适应性",
]

for rec in recommendations:
    print(rec)

## 15.9 本章小结

本章我们学习了：

1. **在线服务架构**：
   - 数据源 → 特征计算 → 模型推理 → 信号生成 → 订单执行

2. **模型管理**：
   - 版本控制、A/B测试、回滚机制

3. **数据更新**：
   - 定时更新、增量更新、校验机制

4. **风险控制**：
   - 仓位限制、行业限制、止损机制

### 下一章预告

下一章我们将学习自定义扩展开发，包括：
- 自定义数据提供器
- 自定义模型
- 自定义策略